# REE Lambda Fitting — Part 2: Lambda Fitting

Fits orthogonal polynomial lambdas (O'Neill 2016) to every sample in
`df_analysis.parquet` and saves the results to `df_model.parquet`.

$$
\ln([\mathrm{REE}_i]_N)
= \lambda_0
+ \lambda_1 f_1(r_i)
+ \lambda_2 f_2(r_i)
+ \lambda_3 f_3(r_i)
+ \lambda_4 f_4(r_i)
$$

| Parameter | Meaning |
|-----------|---------|
| λ₀ | Overall enrichment level |
| λ₁ | LREE–HREE slope |
| λ₂ | Broad curvature |
| λ₃, λ₄ | Higher-order fine-scale variation |

**Inputs**
- `df_analysis.parquet` — from `01_data_prep.ipynb`

**Outputs**
- `df_model.parquet` — `df_analysis` joined with fitted λ values and fit statistics


## 1. Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


## 2. Constants

In [ ]:
REE = ["La", "Ce", "Pr", "Nd", "Sm", "Eu", "Gd", "Tb", "Dy", "Ho", "Er", "Tm", "Yb", "Lu"]

IONIC_RADII = np.array([
    1.160, 1.143, 1.126, 1.109, 1.079, 1.066,
    1.053, 1.040, 1.027, 1.015, 1.004, 0.994,
    0.985, 0.977
], dtype=float)

CI = np.array([
    0.2472, 0.6308, 0.0950, 0.4793, 0.1542, 0.0592,
    0.2059, 0.0375, 0.2540, 0.0554, 0.1645, 0.0258,
    0.1684, 0.0251
], dtype=float)


## 3. Load cleaned data

In [ ]:
df_analysis = pd.read_csv("df_analysis.csv", index_col=0)
print("df_analysis shape:", df_analysis.shape)
df_analysis.head()


## 4. Lambda framework

### 4.1 Orthogonal polynomial basis (O'Neill 2016)

Builds the design matrix **F** whose columns are the shape curves
$f_0, f_1, \ldots, f_d$ evaluated at each ionic radius.


In [ ]:
def oneill_2016_basis(r, degree=4):
    """
    Build the O'Neill (2016) orthogonal polynomial design matrix F.

    Each column is one shape curve evaluated at the ionic radii in `r`:
      f0 = 1            (overall offset)
      f1 = (r - r1)     (linear tilt)
      f2 = (r-r1)(r-r2) (curvature)
      … and so on up to `degree`.

    Parameters
    ----------
    r      : array-like, ionic radii (Å) for the REEs being fitted
    degree : int, highest polynomial order (default 4 → λ0..λ4)

    Returns
    -------
    F : ndarray, shape (len(r), degree+1)
    """
    r = np.asarray(r, dtype=float)

    f0 = np.ones_like(r)
    f1 = (r - 1.05477)
    f2 = (r - 1.00533) * (r - 1.12824)
    f3 = (r - 0.99141) * (r - 1.06055) * (r - 1.14552)
    f4 = (r - 0.98482) * (r - 1.03052) * (r - 1.10441) * (r - 1.15343)

    cols = [f0, f1, f2, f3, f4]
    return np.column_stack(cols[: degree + 1])


### 4.2 Safe log-normalisation

In [ ]:
def _safe_ln_ratio(x, ci):
    """
    Compute ln(x / ci) element-wise, returning NaN wherever either value
    is non-positive, NaN, or infinite.

    Returns
    -------
    y    : ndarray, log-normalised values (NaN where invalid)
    good : boolean mask, True where the calculation was valid
    """
    x  = np.asarray(x,  float)
    ci = np.asarray(ci, float)

    y    = np.full_like(x, np.nan, dtype=float)
    good = np.isfinite(x) & np.isfinite(ci) & (x > 0) & (ci > 0)
    y[good] = np.log(x[good] / ci[good])

    return y, good


### 4.3 OLS solver and covariance

Finds the λ vector that minimises $\|y - F\lambda\|^2$, then estimates
standard errors and the full covariance matrix.


In [ ]:
def fit_lambdas_ols(F, y):
    """Ordinary least-squares: solve F @ beta ≈ y (numpy lstsq wrapper)."""
    beta, *_ = np.linalg.lstsq(F, y, rcond=1e-12)
    return beta


def _ols_cov_se(F, y, beta):
    """
    Compute OLS covariance matrix and per-parameter standard errors.

    Returns (cov, se, dof, s2, resid).
    Returns (None, None, dof, nan, resid) if degrees of freedom <= 0.
    """
    n, p  = F.shape
    resid = y - F @ beta
    dof   = n - p

    if dof <= 0:
        return None, None, dof, np.nan, resid

    sse     = float(np.sum(resid**2))
    s2      = sse / dof
    XtX_inv = np.linalg.pinv(F.T @ F)
    cov     = s2 * XtX_inv
    se      = np.sqrt(np.diag(cov))

    return cov, se, dof, s2, resid


### 4.4 Main fitting function

`lambda_lnREE()` takes one sample's ppm values and returns a dictionary
containing λ₀–λ₄, their uncertainties, and fit-quality statistics.


In [ ]:
def lambda_lnREE(
    sample_ppm,
    degree=4,
    exclude=(),
    s_lnREE_percent=2.0,   # assumed % uncertainty in ln-space (placeholder until per-element σ available)
    ree_order=REE,
    radii=IONIC_RADII,
    ci=CI,
    min_points=None,
):
    """
    Fit orthogonal polynomial lambdas to one REE sample.

    Parameters
    ----------
    sample_ppm     : array-like, REE concentrations (ppm), length = len(ree_order)
    degree         : int, polynomial degree (default 4)
    exclude        : iterable of element names to omit (e.g. ('Eu',) for Eu anomaly)
    s_lnREE_percent: assumed uniform % uncertainty in ln-space, used only for χ²
    ree_order      : list of REE names matching sample_ppm
    radii          : ionic radii array matching ree_order
    ci             : CI normalising values matching ree_order
    min_points     : minimum usable REEs required (default = degree+1)

    Returns
    -------
    dict with keys: lambdas, cov, se, degree, exclude, fit_mask, n_used,
                    n_params, dof, SSE, R2, adj_R2, chi_sq, red_chi_sq,
                    y_obs_used, y_pred_used, resid_used,
                    y_all, y_pred_all, pred_ratio_all, obs_ratio_all, MFR_all
    """
    sample_ppm = np.asarray(sample_ppm, dtype=float)
    radii      = np.asarray(radii,      dtype=float)
    ci         = np.asarray(ci,         dtype=float)

    # (A) Log-normalise
    y_all, good_all = _safe_ln_ratio(sample_ppm, ci)

    # (B) Exclusion mask
    excl_mask = np.ones(len(ree_order), dtype=bool)
    for el in exclude:
        if el in ree_order:
            excl_mask[ree_order.index(el)] = False

    # (C) Final fit mask
    fit_mask = good_all & excl_mask
    rr = radii[fit_mask]
    y  = y_all[fit_mask]

    p = degree + 1
    if min_points is None:
        min_points = p
    if y.size < min_points:
        raise ValueError(
            f"Not enough valid REEs for degree={degree}: "
            f"need >= {min_points}, have {y.size}."
        )

    # (D) Build F and solve
    F       = oneill_2016_basis(rr, degree=degree)
    lambdas = fit_lambdas_ols(F, y)

    # (E) Fit statistics
    y_pred_used = F @ lambdas
    resid_used  = y - y_pred_used
    sse  = float(np.sum(resid_used**2))
    n    = int(y.size)
    dof  = int(n - p)

    y_mean = float(np.mean(y))
    tss    = float(np.sum((y - y_mean)**2))
    R2     = np.nan if tss == 0 else 1.0 - (sse / tss)
    adj_R2 = np.nan
    if dof > 0 and tss != 0:
        adj_R2 = 1.0 - (1.0 - R2) * (n - 1) / dof

    sigma_ln_assumed = 0.01 * float(s_lnREE_percent)
    chi_sq    = float(np.sum((resid_used / sigma_ln_assumed)**2))
    red_chi_sq = np.nan if dof <= 0 else chi_sq / dof

    cov, se, dof2, s2, _ = _ols_cov_se(F, y, lambdas)

    # (F) Predict full 14-REE pattern
    F_all         = oneill_2016_basis(radii, degree=degree)
    y_pred_all    = F_all @ lambdas
    pred_ratio_all = np.exp(np.clip(y_pred_all, -50, 50))

    obs_ratio_all = np.full_like(sample_ppm, np.nan, dtype=float)
    obs_ratio_all[good_all] = sample_ppm[good_all] / ci[good_all]

    MFR_all = np.full_like(sample_ppm, np.nan, dtype=float)
    ok = good_all & np.isfinite(pred_ratio_all) & (pred_ratio_all > 0)
    MFR_all[ok] = obs_ratio_all[ok] / pred_ratio_all[ok]

    return {
        "lambdas": lambdas, "cov": cov, "se": se,
        "degree": degree, "exclude": exclude,
        "fit_mask": fit_mask, "n_used": n, "n_params": p, "dof": dof,
        "SSE": sse, "R2": R2, "adj_R2": adj_R2,
        "chi_sq": chi_sq, "red_chi_sq": red_chi_sq,
        "y_obs_used": y, "y_pred_used": y_pred_used, "resid_used": resid_used,
        "y_all": y_all, "y_pred_all": y_pred_all,
        "pred_ratio_all": pred_ratio_all, "obs_ratio_all": obs_ratio_all,
        "MFR_all": MFR_all,
    }


## 5. Batch fitting

Run `lambda_lnREE()` for every sample and collect results into `df_model`.


In [ ]:
def fit_one_row(row_id, row, REE):
    """Fit one row from df_analysis; return a flat results dict."""
    sample = row[REE].to_numpy(float)
    out    = lambda_lnREE(sample, degree=4, exclude=(), s_lnREE_percent=2.0)

    lam = out["lambdas"]
    se  = out["se"] if out["se"] is not None else [np.nan] * len(lam)

    return {
        "row_id": row_id,
        "λ0": lam[0], "λ1": lam[1], "λ2": lam[2], "λ3": lam[3], "λ4": lam[4],
        "λ0_σ": se[0], "λ1_σ": se[1], "λ2_σ": se[2], "λ3_σ": se[3], "λ4_σ": se[4],
        "SSE": out["SSE"], "R2": out["R2"], "adj_R2": out["adj_R2"],
        "chi_sq": out["chi_sq"], "red_chi_sq": out["red_chi_sq"],
        "n_used": out["n_used"], "dof": out["dof"],
    }


records, failed = [], 0

for i, row in df_analysis.iterrows():
    try:
        records.append(fit_one_row(i, row, REE))
    except Exception:
        failed += 1

df_lambdas = pd.DataFrame(records).set_index("row_id").sort_index()
df_model   = df_analysis.join(df_lambdas, how="inner")

print(f"Total rows in df_analysis : {len(df_analysis)}")
print(f"Rows successfully fitted  : {len(df_model)}")
print(f"Rows failed               : {failed}")


## 6. Fit diagnostics

In [ ]:
print("Distribution of n_used (REEs per fit):")
display(df_model["n_used"].describe())

print("\nRows with all 5 lambdas finite:",
      np.isfinite(df_model[["λ0","λ1","λ2","λ3","λ4"]].to_numpy()).all(axis=1).sum())

print("\nReduced χ² summary:")
display(df_model["red_chi_sq"].describe())


## 7. Single-sample fit check

In [ ]:
# ── Choose a sample to inspect ────────────────────────────────
ROW_ID      = 188    # set to None to pick randomly
ROW_POS     = None   # or set a positional index
RANDOM_SEED = 42

df_one = df_model

if ROW_ID is not None:
    if ROW_ID not in df_one.index:
        raise ValueError(f"ROW_ID={ROW_ID} not found.")
    chosen = ROW_ID
elif ROW_POS is not None:
    chosen = df_one.index[int(ROW_POS)]
else:
    rng    = np.random.default_rng(RANDOM_SEED)
    chosen = rng.choice(df_one.index.to_numpy())

sample_ppm = df_one.loc[chosen, REE].to_numpy(dtype=float)
out        = lambda_lnREE(sample_ppm, degree=4, exclude=(), s_lnREE_percent=2.0)
fit_mask   = out["fit_mask"]

x      = IONIC_RADII[fit_mask]
labels = [e for e, m in zip(REE, fit_mask) if m]

fig, ax = plt.subplots(figsize=(6.2, 4.8))
ax.plot(x, out["y_obs_used"],  marker="o", linewidth=1.2, label="Observed ln(REE/CI)")
ax.plot(x, out["y_pred_used"], marker="o", linestyle="--", linewidth=1.2, label="Lambda fit")
ax.invert_xaxis()
ax.set_xticks(x)
ax.set_xticklabels(labels, rotation=90, ha="center", va="top")
ax.set_xlabel("Ionic radius (Å)")
ax.set_ylabel("ln(REE/CI)")
ax.set_title(
    f"Observed vs Lambda fit\n"
    f"(row_id={chosen}, n_used={out['n_used']}, dof={out['dof']})",
    fontsize=11
)
ax.legend(frameon=False, fontsize=9)
plt.subplots_adjust(bottom=0.25)
plt.show()


## 8. Save output

In [ ]:
df_model.to_csv("df_model.csv", index=True)
print("Saved df_model.csv —", len(df_model), "rows")
